# RAG Agent — Demo

Şirket bilgi tabanı üzerinde Türkçe soru-cevap. Sekiz senaryo, her biri farklı bir
yeteneği kanıtlıyor: XLSX satır retrieval, DOCX başlık hiyerarşisi, **DOCX tablosu**,
PDF bölüm + sayfa atfı, "bilmiyorum" davranışı ve konu dışı filtresi.

**Ön koşul:** `python scripts/ingest.py` çalıştırılmış olmalı (`storage/` dolu).

> Yerel model (Ollama, qwen2.5:7b-instruct) CPU'da çalışıyor; her soru 1–3 dakika sürebilir.

In [1]:
import os
import sys
import time
from pathlib import Path

# DATA_DIR ve STORAGE_DIR göreli yollar. Proje köküne sabitlenmezse notebook
# kendi klasöründe boş bir indeks arar ve "Collection documents does not exist" alır.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.rag.cli import build_agent  # noqa: E402
from src.rag.config import Config  # noqa: E402

config = Config.load()
agent = build_agent(config)

print(f"Sağlayıcı : {config.llm_provider} / {config.llm_model}")
print(f"Embedding : {config.embedding_model}")
print(f"Eşikler   : cosine >= {config.min_cosine}  ve  bm25 >= {config.min_bm25}")
print(f"İndeks    : {len(agent.retriever.index.chunks)} chunk")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Sağlayıcı : ollama / qwen2.5:7b-instruct-q4_K_M
Embedding : intfloat/multilingual-e5-base
Eşikler   : cosine >= 0.8  ve  bm25 >= 5.0
İndeks    : 276 chunk


In [2]:
def sor(soru: str) -> None:
    """Bir soruyu agent'a sorar; cevabı, kaynakları ve tool izini yazdırır."""
    baslangic = time.perf_counter()
    cevap = agent.answer(soru)
    sure = time.perf_counter() - baslangic

    print(f"SORU: {soru}")
    print("-" * 78)
    print(cevap.text)
    print("-" * 78)
    if cevap.citations:
        print("KAYNAKLAR:")
        for sira, etiket in enumerate(cevap.citations, start=1):
            print(f"  {sira}. {etiket}")
    else:
        print("KAYNAKLAR: (yok)")
    print("TOOL İZİ:")
    if cevap.tool_trace:
        for adim in cevap.tool_trace:
            print(
                f"  {adim['name']}({adim['arguments']}) "
                f"-> {adim['chars']} karakter, otomatik={adim['injected']}"
            )
    else:
        print("  (tool çağrılmadı — skor kapısı LLM'den önce devreye girdi)")
    print(f"Süre: {sure:.1f} sn")

## Senaryo 1 — Yıllık izin talebi (XLSX + çoklu kaynak)

In [3]:
sor("Yıllık izin talebimi nasıl yaparım?")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


SORU: Yıllık izin talebimi nasıl yaparım?
------------------------------------------------------------------------------
Yıllık izin talebinizi yapmak için aşağıdaki adımları takip edebilirsiniz:

1. HRPortal (hrportal.teknopark.com.tr) sistemine giriş yapın.
2. 'Izin Talepleri' menüsünü bulun ve "Yıllık İzin" seçeneğini tıklayın.
3. Yeni izin talebi oluşturmak için gerekli bilgileri doldurun (örneğin, başlangıç tarihi, bitiş tarihi).
4. Talebinizi gönderdiğinizde, 2-5 iş günü içinde onaylanması beklenir.

Yıllık izin haklarınızın ne olduğunu öğrenmek için de HRPortal'da 'Izin Bakiyem' widgetini kullanabilir veya IK departmanına Slack kanalı (#ik-destek) üzerinden sorgulayabilirsiniz. 

Kaynaklar:
[1], [3]
------------------------------------------------------------------------------
KAYNAKLAR:
  1. calisan_sss_rehberi.xlsx — Genel SSS, satır 4
  2. calisan_sss_rehberi.xlsx — Genel SSS, satır 5
TOOL İZİ:
  search_documents({'query': 'yıllık izin nasıl yapılır', 'source_filter': '', 'to

## Senaryo 2 — İşe alım süreci (DOCX başlık hiyerarşisi)

In [4]:
sor("İşe alım süreci kaç aşamadan oluşur?")

SORU: İşe alım süreci kaç aşamadan oluşur?
------------------------------------------------------------------------------
İş alım süreci, belgelerde dört aşamadan oluşur:

1. **Ilan Süreci** - Bu aşamada pozisyonun ihtiyaç duyulduğu departman tarafından ilan yapılır.
2. **Mulakat Süreci** - Bu aşamada adaylar değerlendirilir ve mulakatlere çağrılır.
3. **Teklif ve İş Başlangıç Süreci** - En uygun aday için teklif verilir ve işe başlanır.
4. **Oryantasyon Süreci (Pre-Boarding)** - Yeni çalışanın işe başlamadan önce gerekli hazırlıklar yapılır.

Kaynak: [2]
------------------------------------------------------------------------------
KAYNAKLAR:
  1. ik_surecleri_politikası.docx — 2. ISE ALIM SURECI > 2.3 Mulakat Sureci
TOOL İZİ:
  search_documents({'source_filter': '', 'top_k': 5, 'query': 'işe alım süreci kaç aşamadan oluşur'}) -> 2116 karakter, otomatik=False
Süre: 131.1 sn


## Senaryo 3 — Yakıt limiti (**DOCX tablosu**)

`1.500 TL/ay` değeri belgede **yalnızca bir tablonun içinde** geçiyor.
`document.paragraphs` ile okunsaydı bu soru cevapsız kalırdı.

In [5]:
sor("Direktör seviyesindeki bir çalışanın aylık yakıt limiti nedir?")

SORU: Direktör seviyesindeki bir çalışanın aylık yakıt limiti nedir?
------------------------------------------------------------------------------
Direktör seviyesindeki bir çalışanın aylık yakıt limiti 1.500 TL/ay'dır. Bu bilgi [1] numaralı arac kullanım prosedürü belgesinden elde edilmiştir.
------------------------------------------------------------------------------
KAYNAKLAR:
  1. arac_kullanim_proseduru.docx — 3. ARAC TAHSIS POLITIKASI
TOOL İZİ:
  search_documents({'query': 'direktör seviyesindeki çalışan aylık yakıt limiti', 'source_filter': '', 'top_k': 5}) -> 4916 karakter, otomatik=False
Süre: 235.9 sn


## Senaryo 4 — Havuz aracı talebi (DOCX prosedür adımları)

In [6]:
sor("Havuz aracı nasıl talep edilir?")

SORU: Havuz aracı nasıl talep edilir?
------------------------------------------------------------------------------
Havuz aracı talep edilirken şu adımları takip etmelisiniz:

1. FleetApp (filo.teknopark.com.tr) üzerinden talep formunu doldurun:
   - Tarih, saat belirtin
   - Gidilecek yeri yazın
   - Tahmini kilometrajı girin

2. Mudur onayı almak için en az 48 saat önce formu tamamlayın.

3. Arac teslim noktasından (B Blok zemin kat) anahtarı alın ve Teslim Alma Formunu imzalayın.

4. Aracı kullanmadan önce hasar durumunu ve yakıt seviyesini kontrol edin; mevcut hasarlari formda belirtin.

5. Kullanım sonrası aracı tam yakitli halde iade edin ve iade tutanakını imzulayın.

Bu adımları takip ettiğinizde, havuz aracı başarıyla talep edebilirsiniz. Daha fazla bilgiye ihtiyacınız olursa lütfen belgeleri kontrol edin veya ilgili yetkililerle iletişime geçin.

Kaynak: [1]
------------------------------------------------------------------------------
KAYNAKLAR:
  1. arac_kullanim_proseduru

## Senaryo 5 — Aksef kontrendikasyonları (**PDF bölüm + sayfa atfı**)

In [7]:
sor("Aksef 500 mg'ın kontrendikasyonları nelerdir?")

SORU: Aksef 500 mg'ın kontrendikasyonları nelerdir?
------------------------------------------------------------------------------
Aksef 500 mg'ın genel kontrendikasyonları hakkında belirli bir bölüm bulunmamaktadır. Ancak, belgelerde şu bilgiler bulunmaktadır:

- Böbrek yetmezliği olan hastalarda sefuroksim dozunun azaltılması önerilmiştir ([3]).
- Gebelik dönemi ve laktasyon dönemleri için reçete edilmektedir ancak olası yarar riskten fazla olduğunda kullanılabilir ([4]).

Bu bilgiler, Aksef 500 mg'ın genel kontrendikasyonlarını yansıtmaktadır. Daha spesifik kontrendikasyonlar belgenin ilgili bölümlerinde yer almamıştır.
------------------------------------------------------------------------------
KAYNAKLAR:
  1. Aksef 500 mg FKTB_Onaylı KUB.pdf — Bölüm 4.2 Pozoloji ve uygulama şekli, s.1
  2. Aksef 500 mg FKTB_Onaylı KUB.pdf — Bölüm 4.6 Gebelik ve laktasyon Genel tavsiye, s.4
TOOL İZİ:
  search_documents({'query': 'Aksef 500 mg kontrendikasyonları', 'source_filter': 'Aksef', 'top_k

## Senaryo 6 — Duxet gebelikte kullanım (24 sayfalık PDF'te doğru bölüm)

In [8]:
sor("Duxet'in gebelikte kullanımı hakkında ne yazıyor?")

SORU: Duxet'in gebelikte kullanımı hakkında ne yazıyor?
------------------------------------------------------------------------------
Duxet'in gebelikte kullanımı konusunda aşağıdaki noktalar vurgulanmaktadır:

- Gebeliğin son haftalarında Duxet'in kullanımı bebek kesilme semptomları ve diğer sağlık sorunlarına yol açabilir.
- Duxet, fetusun maruz kalabileceği potansiyel risklerden dolayı gebelik döneminde genellikle kullanılmamalıdır. 
- Laktasyon döneminde de Duxetin annenin sütuna salınma olasılığı düşük olduğu belirtilmiştir.
- Gebeliğin son haftalarında Duxet maruziyeti ile ilişkili erken doğum riski artabilir.

Bu noktalar, Duxet'in gebelik ve laktasyon dönemlerindeki kullanım konusunda dikkatli olunması gerektiğini vurgulamaktadır. [1], [2]
------------------------------------------------------------------------------
KAYNAKLAR:
  1. Duxet 30 mg GRSK_Onaylı KUB.pdf — Bölüm 4.6 Gebelik ve laktasyon, s.10
TOOL İZİ:
  search_documents({'source_filter': '', 'top_k': 5, 'query': 'Du

## Senaryo 7 — Ürün taksonomisi (yapılandırılmış XLSX satırı)

In [9]:
sor("Vitatin95 ürününün terapötik sistemi ve ürün müdürü kim?")

SORU: Vitatin95 ürününün terapötik sistemi ve ürün müdürü kim?
------------------------------------------------------------------------------
Vitatin95 ürününün terapötik sistemi kardiyovasküler olup, medikal müdürü MM_19 ve ürün müdürü PM_01'dir. Bu bilgi [1] kaynaklıdır.
------------------------------------------------------------------------------
KAYNAKLAR:
  1. Anonim_Urun_Taksonomi_100Satir.xlsx — Sheet1, satır 2
TOOL İZİ:
  search_documents({'query': 'vitatin95 terapötik sistem ürün müdürü', 'top_k': 5}) -> 2323 karakter, otomatik=False
Süre: 127.9 sn


## Senaryo 8a — "Bilmiyorum" (alan içi ama belgede yok)

Soru şirketle ilgili, ama bilgi tabanında karşılığı yok. Sistem uydurmamalı.

In [10]:
sor("Şirketin 2027 yılı kâr hedefi nedir?")

SORU: Şirketin 2027 yılı kâr hedefi nedir?
------------------------------------------------------------------------------
Bu soru şirket bilgi tabanımın kapsamı dışında görünüyor. Ben İK politikaları, araç kullanım prosedürü, çalışan SSS, ürün taksonomisi ve ilaç kısa ürün bilgisi (KÜB) belgeleri hakkındaki soruları yanıtlayabiliyorum.
------------------------------------------------------------------------------
KAYNAKLAR: (yok)
TOOL İZİ:
  (tool çağrılmadı — skor kapısı LLM'den önce devreye girdi)
Süre: 0.1 sn


## Senaryo 8b — Konu dışı filtresi

Skor kapısı LLM'den **önce** devreye girer: tool çağrısı yapılmaz, LLM hiç çağrılmaz.

In [11]:
sor("Bugün hava nasıl olacak?")

SORU: Bugün hava nasıl olacak?
------------------------------------------------------------------------------
Bu soru şirket bilgi tabanımın kapsamı dışında görünüyor. Ben İK politikaları, araç kullanım prosedürü, çalışan SSS, ürün taksonomisi ve ilaç kısa ürün bilgisi (KÜB) belgeleri hakkındaki soruları yanıtlayabiliyorum.
------------------------------------------------------------------------------
KAYNAKLAR: (yok)
TOOL İZİ:
  (tool çağrılmadı — skor kapısı LLM'den önce devreye girdi)
Süre: 0.1 sn


## Eşik kalibrasyonu — ölçülmüş skor dağılımı

Güven kapısı `cosine >= 0.80` **ve** `bm25 >= 5.0` şartını arar. Aşağıdaki tablo bu iki
eşiğin neden birlikte gerektiğini gösteriyor: tek başına hiçbiri geçerli soruları konu
dışı olanlardan ayıramıyor (bantlar çakışıyor).

In [12]:
GECERLI = [
    "Yıllık izin talebimi nasıl yaparım?",
    "İşe alım süreci kaç aşamadan oluşur?",
    "Direktör seviyesindeki bir çalışanın aylık yakıt limiti nedir?",
    "Havuz aracı nasıl talep edilir?",
    "Aksef 500 mg'ın kontrendikasyonları nelerdir?",
    "Duxet'in gebelikte kullanımı hakkında ne yazıyor?",
    "Vitatin95 ürününün terapötik sistemi ve ürün müdürü kim?",
]
KONU_DISI = [
    "Bugün hava nasıl olacak?",
    "En sevdiğin film hangisi?",
    "Bitcoin fiyatı ne kadar?",
    "Türkiye'nin başkenti neresidir?",
    "Bana bir şiir yaz",
    "Şirketin 2027 yılı kâr hedefi nedir?",
]


def skorlar(sorular, etiket):
    print(f"\n{etiket}")
    print(f"{'kosinüs':>8} {'bm25':>7}  {'kapı':>6}  soru")
    for soru in sorular:
        sonuclar = agent.retriever.search(soru, top_k=5)
        en_iyi = sonuclar[0]
        gecti = "GEÇTİ" if agent.retriever.is_confident(sonuclar) else "ELENDİ"
        print(f"{en_iyi.cosine:8.3f} {en_iyi.bm25:7.2f}  {gecti:>6}  {soru}")


skorlar(GECERLI, "GEÇERLİ SORULAR (hepsi geçmeli)")
skorlar(KONU_DISI, "KONU DIŞI / BELGEDE YOK (hiçbiri geçmemeli)")


GEÇERLİ SORULAR (hepsi geçmeli)
 kosinüs    bm25    kapı  soru


   0.874   21.55   GEÇTİ  Yıllık izin talebimi nasıl yaparım?
   0.813   15.03   GEÇTİ  İşe alım süreci kaç aşamadan oluşur?


   0.833   12.01   GEÇTİ  Direktör seviyesindeki bir çalışanın aylık yakıt limiti nedir?


   0.861   17.04   GEÇTİ  Havuz aracı nasıl talep edilir?
   0.832   13.08   GEÇTİ  Aksef 500 mg'ın kontrendikasyonları nelerdir?


   0.849    7.51   GEÇTİ  Duxet'in gebelikte kullanımı hakkında ne yazıyor?


   0.867    7.68   GEÇTİ  Vitatin95 ürününün terapötik sistemi ve ürün müdürü kim?

KONU DIŞI / BELGEDE YOK (hiçbiri geçmemeli)
 kosinüs    bm25    kapı  soru
   0.782    3.64  ELENDİ  Bugün hava nasıl olacak?
   0.746    8.25  ELENDİ  En sevdiğin film hangisi?


   0.782    4.51  ELENDİ  Bitcoin fiyatı ne kadar?


   0.776    3.20  ELENDİ  Türkiye'nin başkenti neresidir?
   0.813    0.00  ELENDİ  Bana bir şiir yaz
   0.778    4.09  ELENDİ  Şirketin 2027 yılı kâr hedefi nedir?
